# CAATs de análisis de datos — 2. Pruebas analíticas y detección de fraude

**Auditoría Informática — LSI — FCEQyN — UNaM**

Con los datos ya evaluados (notebook 1), aplicamos pruebas sobre el **100% de la población** para detectar errores, incumplimientos de controles y posibles fraudes en el ciclo de compras y pagos de **Yerbatera del Alto Paraná S.A.** (ejercicio 2025).

### Políticas de la empresa (relevadas en entrevistas)

1. Los pagos de **$ 500.000 o más** requieren aprobación adicional de la gerencia.
2. Tesorería opera de **lunes a viernes, de 8 a 17 h**.
3. **Segregación de funciones:** Compras carga las facturas; Tesorería aprueba los pagos. Una misma persona no puede hacer las dos cosas.
4. Los pagos se transfieren **únicamente al CBU registrado** en el maestro de proveedores.
5. Las órdenes de pago se numeran en forma **correlativa** y no se reutilizan.

| Prueba | ACL / IDEA | pandas |
|---|---|---|
| Estadísticas, estratificación | *Statistics*, *Stratify* | `describe()`, `pd.cut()` + `groupby()` |
| Resumen por proveedor | *Summarize* | `groupby().agg()` |
| Faltantes en secuencias | *Gaps* | `diff()` sobre la numeración |
| Duplicados | *Duplicates* | `duplicated()` |
| Cruce de archivos | *Join* / *Relate* | `merge()` |
| Ley de Benford | *Benford* | cálculo propio + `scipy.stats` |
| Antigüedad / fechas | *Age* | `.dt.hour`, `.dt.dayofweek` |
| Muestreo | *Sample* (aleatorio, MUS) | `sample()`, cálculo propio |

## 0. Preparación

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

DATOS = Path("datos")
RESULTADOS = Path("resultados")
RESULTADOS.mkdir(exist_ok=True)
UMBRAL = 500_000

empleados = pd.read_csv(DATOS / "empleados.csv", dtype=str, parse_dates=["fecha_ingreso", "fecha_baja"])
proveedores = pd.read_csv(DATOS / "proveedores.csv", dtype=str, parse_dates=["fecha_alta"])
facturas = pd.read_csv(DATOS / "facturas.csv", dtype=str)
pagos = pd.read_csv(DATOS / "pagos.csv", dtype=str, parse_dates=["fecha_hora"])

for col in ["importe_neto", "iva", "total"]:
    facturas[col] = pd.to_numeric(facturas[col])
pagos["importe"] = pd.to_numeric(pagos["importe"])
facturas["fecha"] = pd.to_datetime(facturas.fecha, format="%Y-%m-%d", errors="coerce")

hallazgos = {}


def registrar(nombre, excepciones):
    hallazgos[nombre] = excepciones
    print(f"{nombre}: {len(excepciones)} excepciones")
    return excepciones

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter, ScalarFormatter

# estilo sobrio: grilla tenue, sin bordes superiores/derechos
plt.rcParams.update({
    "figure.figsize": (9, 4), "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": "#52514e", "axes.titlecolor": "#0b0b0b",
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "axes.grid.axis": "y",
    "axes.axisbelow": True,
    "grid.color": "#e6e5e0", "grid.linewidth": 0.8, "xtick.color": "#52514e", "ytick.color": "#52514e",
    "axes.titlesize": 12, "axes.titleweight": "bold", "font.size": 10,
})
AZUL, NARANJA = "#2a78d6", "#eb6834"

## 1. Conocer la población

### 1.1 Estadísticas generales de los pagos

In [ ]:
pagos.importe.describe().apply(lambda x: f"{x:,.2f}")

In [ ]:
print(f"Total pagado en el período: $ {pagos.importe.sum():,.2f} en {len(pagos)} órdenes de pago")

### 1.2 Estratificación

Agrupar los pagos por rango de importe muestra dónde se concentra el **valor** (no la cantidad). Es la base para decidir la estrategia de muestreo.

In [ ]:
cortes = [0, 10_000, 50_000, 100_000, 250_000, UMBRAL, 1_000_000, 5_000_000, np.inf]
etiquetas = ["< 10 mil", "10–50 mil", "50–100 mil", "100–250 mil", "250–500 mil", "500 mil–1 M", "1–5 M", "> 5 M"]
pagos["estrato"] = pd.cut(pagos.importe, cortes, labels=etiquetas, right=False)

estratos = pagos.groupby("estrato", observed=False).importe.agg(cantidad="count", importe="sum")
estratos["% cantidad"] = (estratos.cantidad / estratos.cantidad.sum() * 100).round(1)
estratos["% importe"] = (estratos.importe / estratos.importe.sum() * 100).round(1)
estratos.style.format({"importe": "{:,.0f}"})

### 1.3 Concentración por proveedor

In [ ]:
por_proveedor = (pagos.groupby("id_proveedor").importe.agg(pagos="count", total="sum")
                 .sort_values("total", ascending=False)
                 .join(proveedores.set_index("id_proveedor")[["razon_social", "fecha_alta"]]))
por_proveedor["% acumulado"] = (por_proveedor.total.cumsum() / por_proveedor.total.sum() * 100).round(1)
por_proveedor.head(10).style.format({"total": "{:,.0f}"})

## 2. Secuencias: faltantes y duplicados en órdenes de pago

Una orden de pago faltante puede ser un comprobante anulado sin documentar o un pago eliminado del sistema. Una repetida, un pago registrado dos veces o una manipulación.

In [ ]:
numero = pagos.nro_orden_pago.str.replace("OP-", "").astype(int).sort_values()
salto = numero.diff()

anterior = numero.shift()
faltantes = [f"OP-{n:06d}" for desde, hasta in zip(anterior[salto > 1], numero[salto > 1])
             for n in range(int(desde) + 1, int(hasta))]
registrar("Secuencia - órdenes de pago faltantes", pd.DataFrame({"nro_orden_pago": faltantes}))

In [ ]:
registrar("Secuencia - número de orden de pago repetido",
          pagos[pagos.nro_orden_pago.duplicated(keep=False)])

## 3. Pagos duplicados

### 3.1 La misma factura pagada más de una vez

In [ ]:
mas_de_un_pago = pagos[pagos.id_factura.duplicated(keep=False)].sort_values(["id_factura", "fecha_hora"])
registrar("Duplicados - factura pagada más de una vez", mas_de_un_pago.drop(columns="estrato"))

### 3.2 La misma factura cargada dos veces (y pagada dos veces)

El notebook 1 encontró facturas cargadas dos veces con distinto código interno. La prueba 3.1 no las detecta porque cada copia tiene su propio `id_factura`. Cruzamos por el comprobante:

In [ ]:
clave = ["id_proveedor", "tipo", "punto_venta", "numero"]
pagos_comprobante = pagos.merge(facturas[["id_factura", *clave[1:]]], on="id_factura")
doble = pagos_comprobante[pagos_comprobante.duplicated(clave, keep=False)].sort_values(clave)
registrar("Duplicados - comprobante pagado más de una vez",
          doble[["nro_orden_pago", "id_factura", *clave, "fecha_hora", "importe"]])

## 4. Pagos vs. facturas

In [ ]:
pf = pagos.merge(facturas[["id_factura", "fecha", "total", "estado", "usuario_carga"]],
                 on="id_factura", how="left")

registrar("Pagos vs facturas - pago mayor que la factura",
          pf[pf.importe > pf.total + 0.01][["nro_orden_pago", "id_factura", "id_proveedor", "importe", "total"]]
          .assign(exceso=lambda d: d.importe - d.total))

In [ ]:
registrar("Pagos vs facturas - pago anterior a la fecha de la factura",
          pf[pf.fecha_hora.dt.normalize() < pf.fecha][["nro_orden_pago", "id_factura", "fecha", "fecha_hora", "importe"]])

## 5. Ley de Benford

En muchos conjuntos de importes "naturales" (facturas, pagos, ventas) el primer dígito **no** es uniforme: el 1 aparece ~30% de las veces y el 9 menos del 5%. La proporción esperada es

$$P(d) = \log_{10}\left(1 + \frac{1}{d}\right)$$

Los importes inventados por personas suelen apartarse de esa distribución. La prueba **no prueba un fraude**: señala dónde mirar.

Medidas (según Nigrini): el **MAD** (desvío absoluto medio) para la conformidad general, y el **estadístico z** por dígito para detectar picos puntuales (|z| > 1,96 ⇒ significativo al 5%).

In [ ]:
def benford(importes, digitos=1):
    """Compara los primeros `digitos` dígitos con la ley de Benford."""
    importes = importes[importes >= 10 ** digitos]
    primeros = importes.astype(int).astype(str).str[:digitos].astype(int)
    rango = range(10 ** (digitos - 1), 10 ** digitos)
    n = len(primeros)
    tabla = pd.DataFrame(index=pd.Index(rango, name="dígito"))
    tabla["cantidad"] = primeros.value_counts().reindex(rango, fill_value=0)
    tabla["observado"] = tabla.cantidad / n
    tabla["esperado"] = np.log10(1 + 1 / tabla.index.to_numpy())
    tabla["z"] = ((tabla.observado - tabla.esperado).abs() - 1 / (2 * n)) / np.sqrt(
        tabla.esperado * (1 - tabla.esperado) / n)
    mad = (tabla.observado - tabla.esperado).abs().mean()
    chi2 = stats.chisquare(tabla.cantidad, tabla.esperado * n)
    return tabla, mad, chi2.pvalue


def grafico_benford(tabla, titulo):
    fig, ax = plt.subplots()
    ancho = 0.7 if len(tabla) < 20 else 0.8
    ax.bar(tabla.index, tabla.observado, width=ancho, color=AZUL, label="Observado")
    ax.plot(tabla.index, tabla.esperado, color=NARANJA, linewidth=2, marker="o" if len(tabla) < 20 else None,
            markersize=6, label="Esperado (Benford)")
    ax.set_title(titulo)
    ax.set_xlabel("Primer dígito" if len(tabla) < 20 else "Primeros dos dígitos")
    ax.set_ylabel("Proporción")
    ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0 if len(tabla) < 20 else 1))
    ax.set_xticks(tabla.index if len(tabla) < 20 else range(10, 100, 5))
    ax.legend(frameon=False)
    return ax

In [ ]:
tabla1, mad1, p1 = benford(facturas.total.dropna())
grafico_benford(tabla1, "Facturas 2025: primer dígito del total")
print(f"MAD = {mad1:.4f}  (conformidad: < 0,006 cercana | 0,006–0,012 aceptable | 0,012–0,015 marginal | > 0,015 no conforme)")
print(f"chi² p-valor = {p1:.4f}")
tabla1.round(4)

El MAD dice que la distribución general es aceptable, pero el **dígito 4** tiene un z muy alto. La prueba de los primeros dos dígitos tiene más resolución:

In [ ]:
tabla2, mad2, p2 = benford(facturas.total.dropna(), digitos=2)
ax = grafico_benford(tabla2, "Facturas 2025: primeros dos dígitos del total")
picos = tabla2[(tabla2.z > 1.96) & (tabla2.observado > tabla2.esperado)].sort_values("z", ascending=False)
for d in picos.index[:3]:
    ax.annotate(str(d), (d, tabla2.observado[d]), textcoords="offset points", xytext=(0, 4),
                ha="center", color="#0b0b0b", fontsize=9)
print(f"MAD = {mad2:.4f}  (primeros dos dígitos: < 0,0012 cercana | hasta 0,0018 aceptable | hasta 0,0022 marginal)")
picos.round(4).head(8)

**Drill-down:** ¿quién emite las facturas que empiezan con 47, 48 y 49?

In [ ]:
facturas["dos_digitos"] = facturas.total.astype(int).astype(str).str[:2]
sospechosas = facturas[facturas.dos_digitos.isin(["47", "48", "49"]) & facturas.total.between(100_000, 999_999)]
(sospechosas.groupby("id_proveedor").total.agg(facturas="count", total="sum")
 .join(proveedores.set_index("id_proveedor")[["razon_social", "fecha_alta", "usuario_alta"]])
 .sort_values("facturas", ascending=False).head(6))

Dos proveedores concentran los picos: ambos dados de **alta en 2025** y por el **mismo usuario**. Las pruebas siguientes los rodean desde otros ángulos.

## 6. Montos cerca del umbral de aprobación y fraccionamiento

### 6.1 Facturas entre el 90% y el 100% del umbral

In [ ]:
cerca = facturas[facturas.total.between(UMBRAL * 0.9, UMBRAL, inclusive="left")]
resumen_cerca = (cerca.groupby("id_proveedor").total.agg(facturas_cerca_umbral="count")
                 .join(facturas.groupby("id_proveedor").total.agg(facturas_totales="count")))
resumen_cerca["proporción"] = (resumen_cerca.facturas_cerca_umbral / resumen_cerca.facturas_totales).round(2)
registrar("Umbral - proveedores con >= 50% de facturas justo debajo del umbral",
          resumen_cerca[(resumen_cerca.proporción >= 0.5) & (resumen_cerca.facturas_totales >= 3)].sort_values("facturas_cerca_umbral", ascending=False).reset_index())

### 6.2 Fraccionamiento: varias facturas del mismo proveedor y día que juntas superan el umbral

Consideramos solo facturas de entre el 50% y el 100% del umbral: dos facturas chicas el mismo día son normales en un proveedor habitual.

In [ ]:
mismo_dia = (facturas[facturas.total.between(UMBRAL * 0.5, UMBRAL, inclusive="left")]
             .groupby(["id_proveedor", "fecha"]).total.agg(facturas="count", suma="sum").reset_index())
registrar("Umbral - posible fraccionamiento (mismo proveedor y día)",
          mismo_dia[(mismo_dia.facturas >= 2) & (mismo_dia.suma >= UMBRAL)].sort_values("suma", ascending=False))

### 6.3 Importes redondos

Los importes reales con IVA rara vez son múltiplos exactos de $ 10.000.

In [ ]:
redondos = facturas[(facturas.total >= 10_000) & (facturas.total % 10_000 == 0)]
registrar("Umbral - facturas con importe redondo", redondos[["id_factura", "id_proveedor", "fecha", "total", "usuario_carga"]])

## 7. Pagos fuera de horario o en fin de semana

In [ ]:
pagos["dia_semana"] = pagos.fecha_hora.dt.dayofweek  # 0 = lunes ... 6 = domingo
pagos["hora"] = pagos.fecha_hora.dt.hour
fuera = pagos[(pagos.dia_semana >= 5) | (pagos.hora < 8) | (pagos.hora >= 17)]
registrar("Horario - pagos fuera del horario de Tesorería",
          fuera[["nro_orden_pago", "id_proveedor", "fecha_hora", "importe", "usuario_aprueba"]])

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
por_hora = pagos.hora.value_counts().reindex(range(24), fill_value=0)
colores = [NARANJA if (h < 8 or h >= 17) else AZUL for h in range(24)]
ax.bar(por_hora.index, por_hora.values, color=colores, width=0.8)
for h, n in por_hora.items():
    if n and (h < 8 or h >= 17):
        ax.annotate(str(n), (h, n), textcoords="offset points", xytext=(0, 3), ha="center", fontsize=9)
ax.set_yscale("log")  # escala logarítmica: si no, los pocos pagos fuera de horario no se ven
ax.yaxis.set_major_formatter(ScalarFormatter())
ax.set_title("Pagos por hora del día (naranja: antes de las 8 o después de las 17; escala logarítmica)")
ax.set_xlabel("Hora")
ax.set_ylabel("Pagos")
ax.set_xticks(range(0, 24, 2))
fuera.usuario_aprueba.value_counts()

El gráfico solo muestra la hora: los pagos de **sábado** en horario "normal" no aparecen en naranja, pero sí en la tabla de excepciones. Siempre revisar la lista completa, no solo el gráfico.

## 8. Segregación de funciones y accesos

### 8.1 La misma persona carga la factura y aprueba el pago

In [ ]:
registrar("Segregación - mismo usuario carga y aprueba",
          pf[pf.usuario_carga == pf.usuario_aprueba][["nro_orden_pago", "id_factura", "id_proveedor", "importe", "usuario_carga", "usuario_aprueba"]])

### 8.2 Aprobaciones por usuarios que no son de Tesorería o que ya no trabajan en la empresa

In [ ]:
aprob = pagos.merge(empleados[["usuario", "apellido", "nombre", "area", "fecha_baja"]],
                    left_on="usuario_aprueba", right_on="usuario", how="left")
registrar("Accesos - aprobación por usuario fuera de Tesorería",
          aprob[aprob.area != "Tesorería"][["nro_orden_pago", "fecha_hora", "importe", "usuario_aprueba", "area"]])
registrar("Accesos - aprobación posterior a la baja del usuario",
          aprob[aprob.fecha_hora.dt.normalize() > aprob.fecha_baja][["nro_orden_pago", "fecha_hora", "importe", "usuario_aprueba", "fecha_baja"]])

Un usuario dado de baja que sigue operando indica que **no se deshabilitó su acceso** (o que alguien usa sus credenciales). Es un hallazgo de control de accesos, típico de auditoría de sistemas.

## 9. Cruces entre archivos

### 9.1 Pagos a un CBU distinto del registrado

In [ ]:
pp = pagos.merge(proveedores[["id_proveedor", "razon_social", "cbu"]], on="id_proveedor", how="left")
otro_cbu = pp[pp.cbu_destino != pp.cbu]
# ¿el CBU de destino pertenece a un empleado?
otro_cbu = otro_cbu.merge(empleados[["cbu", "usuario", "apellido"]].rename(columns={"cbu": "cbu_destino"}),
                          on="cbu_destino", how="left")
registrar("Cruce - pago a CBU distinto del registrado",
          otro_cbu[["nro_orden_pago", "id_proveedor", "razon_social", "importe", "cbu", "cbu_destino", "usuario", "apellido"]])

### 9.2 Proveedores que comparten datos con empleados

Un proveedor con el CBU, el domicilio o el CUIT de un empleado es la señal clásica de un **proveedor fantasma**.

In [ ]:
coincidencias = []
for campo in ["cbu", "domicilio", "cuit"]:
    cruce = proveedores.merge(empleados, on=campo, suffixes=("_prov", "_emp"))
    for _, fila in cruce.iterrows():
        coincidencias.append({"campo": campo, "valor": fila[campo], "id_proveedor": fila.id_proveedor,
                              "razon_social": fila.razon_social, "legajo": fila.legajo,
                              "empleado": f"{fila.apellido}, {fila.nombre}", "area": fila.area})
registrar("Cruce - proveedor con datos de un empleado", pd.DataFrame(coincidencias))

### 9.3 Proveedores dados de alta en el período: volumen y quién los dio de alta

In [ ]:
nuevos = (proveedores[proveedores.fecha_alta >= "2025-01-01"]
          .merge(por_proveedor[["pagos", "total"]], left_on="id_proveedor", right_index=True, how="left")
          .fillna({"pagos": 0, "total": 0})
          .sort_values("total", ascending=False))
nuevos[["id_proveedor", "razon_social", "fecha_alta", "usuario_alta", "pagos", "total"]].style.format({"total": "{:,.0f}"})

## 10. Integración de resultados: ¿qué registros aparecen en varias pruebas?

Una excepción aislada puede ser un error. Varias pruebas que señalan al mismo proveedor o usuario forman un **patrón**.

In [ ]:
señales = []
for nombre, df in hallazgos.items():
    for col in ["id_proveedor", "usuario_aprueba", "usuario_carga"]:
        if col in df.columns:
            for valor in df[col].dropna().unique():
                señales.append({"entidad": valor, "prueba": nombre})
señales = pd.DataFrame(señales).drop_duplicates()
(señales.groupby("entidad").prueba.agg(pruebas="count", detalle=lambda s: " | ".join(sorted(s)))
 .sort_values("pruebas", ascending=False).head(8))

## 11. Muestreo

Las pruebas anteriores cubren el 100% de la población, pero algunos procedimientos (verificar el comprobante físico, confirmar con el proveedor) solo pueden hacerse sobre una muestra.

### 11.1 Muestreo aleatorio simple

Fijar la semilla (`random_state`) hace la muestra **reproducible**: otro auditor obtiene exactamente los mismos ítems. La semilla se documenta en el papel de trabajo.

In [ ]:
SEMILLA = 20251231
muestra_aleatoria = pagos.sample(n=30, random_state=SEMILLA)
print(f"Importe cubierto: $ {muestra_aleatoria.importe.sum():,.0f} "
      f"({muestra_aleatoria.importe.sum() / pagos.importe.sum():.1%} del total)")

### 11.2 Muestreo por unidad monetaria (MUS)

Cada **peso** tiene la misma probabilidad de ser elegido, así que los pagos grandes tienen más chance de entrar en la muestra. Se calcula un intervalo `I = total / n`, se elige un punto de arranque aleatorio entre 0 e `I` y se toma el pago que contiene cada punto `arranque + k·I`. Los pagos mayores que `I` salen siempre.

In [ ]:
def muestreo_mus(df, columna, n, semilla):
    poblacion = df.sort_values("nro_orden_pago").reset_index(drop=True)
    acumulado = poblacion[columna].cumsum()
    intervalo = acumulado.iloc[-1] / n
    arranque = np.random.default_rng(semilla).uniform(0, intervalo)
    puntos = arranque + intervalo * np.arange(n)
    posiciones = np.searchsorted(acumulado.to_numpy(), puntos)
    print(f"Intervalo: $ {intervalo:,.2f} | arranque: $ {arranque:,.2f}")
    return poblacion.iloc[np.unique(posiciones)]


muestra_mus = muestreo_mus(pagos, "importe", n=30, semilla=SEMILLA)
print(f"{len(muestra_mus)} pagos distintos; importe cubierto: $ {muestra_mus.importe.sum():,.0f} "
      f"({muestra_mus.importe.sum() / pagos.importe.sum():.1%} del total)")

## 12. Papel de trabajo

In [ ]:
resumen = pd.DataFrame([(n.split(" - ")[0], n.split(" - ")[1], len(df)) for n, df in hallazgos.items()],
                       columns=["categoría", "prueba", "excepciones"])
archivo = RESULTADOS / "02_pruebas_analiticas.xlsx"
with pd.ExcelWriter(archivo) as excel:
    resumen.to_excel(excel, sheet_name="Resumen", index=False)
    for i, (nombre, df) in enumerate(hallazgos.items(), start=1):
        df.to_excel(excel, sheet_name=f"{i:02d}", index=False)
    muestra_aleatoria.to_excel(excel, sheet_name="Muestra aleatoria", index=False)
    muestra_mus.to_excel(excel, sheet_name="Muestra MUS", index=False)
print(f"Papel de trabajo: {archivo}")
resumen

## 13. Ejercicios

1. **Informe.** Redactar los 3 hallazgos más graves con el formato *condición – criterio – causa – efecto – recomendación*. Para el "efecto", cuantificar el importe involucrado.
2. **Relato del fraude.** Con las pruebas 5, 6, 7, 8 y 9, reconstruir qué pasó con los proveedores `P0119` y `P0120`: quién los dio de alta, quién cargó sus facturas, quién aprobó los pagos y a dónde fue el dinero.
3. **Falsos positivos.** El proveedor señalado por fraccionamiento (6.2), ¿es necesariamente un fraude? ¿Qué documentación pedirían para descartarlo?
4. **Feriados.** Ampliar la prueba 7 para incluir los feriados nacionales de 2025 (24/3, 2/4, 18/4, 1/5, 2/5, 25/5, 16/6, 20/6, 9/7, 15/8, 17/8, 10/10, 12/10, 21/11, 24/11, 8/12, 25/12).
5. **Benford sobre pagos.** Repetir la prueba 5 sobre `pagos.importe`. ¿Cambian las conclusiones?
6. **Muestreo.** Comparar qué proporción de los pagos de `P0119` y `P0120` entra en la muestra aleatoria y en la MUS. ¿Qué método habría detectado el problema sin las pruebas sobre el 100%?
7. **Proveedores sin actividad.** Listar proveedores del maestro que no recibieron pagos en 2025. ¿Por qué un auditor recomendaría depurarlos?